# Napari ROI export for the Cellpose GUI

Opens the large H&E whole-slide TIFF in napari, lets you draw one or more
ROI rectangles, and exports each one as its own `.tif` crop. Use these
crops with the **Cellpose GUI** (not this notebook) to correct masks by
hand and fine-tune `cpsam_v2` — see the instructions at the bottom.

**One-time setup**: run the cell below once in the `cellpose` conda env,
then restart the kernel.


In [ ]:
# One-time setup — uncomment and run once, then restart the kernel.
# %pip install "napari[all]" zarr dask


In [6]:
from pathlib import Path

import numpy as np
import tifffile
import zarr
import dask.array as da
import napari
import matplotlib.pyplot as plt


## 1. Point at the slide

The TIFF is ~9.6 GB uncompressed and strip-compressed (not tiled), so it's
opened lazily through `tifffile`'s zarr store instead of being read into
memory. A quick on-the-fly multiscale pyramid (strided dask slicing) is
handed to napari so it never has to pull the full-resolution array into a
single GPU texture.


In [7]:
TIFF_PATH = "260427_CO36_RCLB5_VisiumHD_20x_5_cropped.tif"

store = tifffile.imread(TIFF_PATH, aszarr=True)
z = zarr.open(store, mode="r")
full_res = da.from_zarr(z)
print(f"Full-resolution shape: {full_res.shape}, dtype: {full_res.dtype}")


Full-resolution shape: (49900, 64335, 3), dtype: uint8


In [8]:
def build_pyramid(arr, n_levels=6, downsample=2):
    """Cheap strided-slice pyramid — good enough for interactive viewing."""
    pyramid = [arr]
    current = arr
    for _ in range(n_levels - 1):
        current = current[::downsample, ::downsample, :]
        pyramid.append(current)
    return pyramid

pyramid = build_pyramid(full_res)
for i, lvl in enumerate(pyramid):
    print(f"level {i}: {lvl.shape}")


level 0: (49900, 64335, 3)
level 1: (24950, 32168, 3)
level 2: (12475, 16084, 3)
level 3: (6238, 8042, 3)
level 4: (3119, 4021, 3)
level 5: (1560, 2011, 3)


## 2. Open napari and draw ROIs

Run the cell below to launch the viewer. A `Shapes` layer named **"ROI"**
is added and selected with the rectangle tool active. Draw as many
rectangles as you want training crops — spread them across different
tissue regions / cell densities for a more useful training set. Each one
becomes a separate exported file in the next step.


In [9]:
viewer = napari.Viewer()
viewer.add_image(
    pyramid,
    multiscale=True,
    rgb=True,
    name="H&E slide",
    contrast_limits=[0, 255],
)

roi_layer = viewer.add_shapes(
    name="ROI",
    edge_color="red",
    face_color="transparent",
    edge_width=20,
)
viewer.layers.selection.active = roi_layer
roi_layer.mode = "add_rectangle"

print("Draw one or more rectangles on the 'ROI' layer, then run the next cell.")


Draw one or more rectangles on the 'ROI' layer, then run the next cell.


## 3. Crop and export each ROI

Run this *after* drawing your rectangles in napari. Each shape on the
`ROI` layer is converted to a pixel bounding box in full-res coordinates,
cropped out of the slide, and saved as its own `.tif` file — ready to
open directly in the Cellpose GUI.


In [11]:
OUTPUT_DIR = Path("cellpose_test_crops")
OUTPUT_DIR.mkdir(exist_ok=True)

SLIDE_STEM = Path(TIFF_PATH).stem

assert len(roi_layer.data) > 0, "Draw at least one rectangle on the 'ROI' layer in napari before running this cell."

exported = []
for i, roi in enumerate(roi_layer.data):
    ys, xs = roi[:, 0], roi[:, 1]

    y0, y1 = int(np.floor(ys.min())), int(np.ceil(ys.max()))
    x0, x1 = int(np.floor(xs.min())), int(np.ceil(xs.max()))
    y0, x0 = max(y0, 0), max(x0, 0)
    y1, x1 = min(y1, full_res.shape[0]), min(x1, full_res.shape[1])

    crop = full_res[y0:y1, x0:x1, :].compute()

    fname = OUTPUT_DIR / f"{SLIDE_STEM}_roi{i:02d}_y{y0}-{y1}_x{x0}-{x1}.tif"
    tifffile.imwrite(fname, crop)
    exported.append(fname)

    print(f"ROI {i}: y[{y0}:{y1}], x[{x0}:{x1}]  ({y1 - y0} x {x1 - x0} px) -> {fname}")

print(f"\nExported {len(exported)} crop(s) to {OUTPUT_DIR.resolve()}")


ROI 0: y[27923:30913], x[4280:6269]  (2990 x 1989 px) -> cellpose_test_crops/260427_CO36_RCLB5_VisiumHD_20x_5_cropped_roi00_y27923-30913_x4280-6269.tif
ROI 1: y[25156:27753], x[4436:7292]  (2597 x 2856 px) -> cellpose_test_crops/260427_CO36_RCLB5_VisiumHD_20x_5_cropped_roi01_y25156-27753_x4436-7292.tif
ROI 2: y[31879:33601], x[3851:5696]  (1722 x 1845 px) -> cellpose_test_crops/260427_CO36_RCLB5_VisiumHD_20x_5_cropped_roi02_y31879-33601_x3851-5696.tif
ROI 3: y[30482:32833], x[8014:11518]  (2351 x 3504 px) -> cellpose_test_crops/260427_CO36_RCLB5_VisiumHD_20x_5_cropped_roi03_y30482-32833_x8014-11518.tif

Exported 4 crop(s) to /Users/sebgoti/Documents/PhD/Celine_brain_segmentation/cellpose_test_crops


## 4. (Optional) Preview the exported crops

In [ ]:
n = len(exported)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5), squeeze=False)
for ax, fname in zip(axes[0], exported):
    ax.imshow(tifffile.imread(fname))
    ax.set_title(fname.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Next: open these crops in the Cellpose GUI

See the instructions provided alongside this notebook for how to load
`cellpose_training_crops/`, correct masks, and train a custom model in
the Cellpose GUI.
